# Proyek: Deteksi Anomali Spektrum RF Cerdas
## 02 - Rekayasa Fitur (Feature Engineering)

**Tujuan Notebook:** Mengubah data spektrum mentah menjadi sekumpulan fitur numerik yang terstruktur. Proses ini adalah jantung dari proyek, di mana kita menerjemahkan pengetahuan domain (fisika sinyal) ke dalam representasi yang dapat dipahami oleh model machine learning.

### Konsep: Dari Sampel ke Karakteristik

Data mentah dari SDR berisi ribuan titik power vs. frekuensi. Ini terlalu granular dan 'berisik' untuk dianalisis secara langsung. Sebaliknya, kita akan mengekstrak karakteristik level-makro dari setiap pengukuran, seperti:

- **Noise Floor:** Tingkat daya rata-rata dari latar belakang spektrum.
- **Sinyal Terdeteksi:** Mengidentifikasi 'puncak' signifikan yang menonjol di atas noise floor.
- **Karakteristik Sinyal Terkuat:** Untuk setiap pengukuran, kita fokus pada sinyal yang paling dominan dan mencatat frekuensi, kekuatan, dan bandwidth-nya.

Pendekatan ini secara drastis mengurangi dimensi data sambil mempertahankan informasi yang paling relevan untuk membedakan satu kondisi spektrum dari yang lain.

In [1]:
import pandas as pd
import numpy as np
import os
import glob
from scipy.signal import find_peaks

### Fungsi Ekstraksi Fitur

In [2]:
def extract_physical_features(file_path, prominence=5, width=1):
    """Mengekstrak fitur fisis dari satu file pengukuran mentah."""
    df = pd.read_csv(file_path, header=None, usecols=[2, 6], names=['frequency', 'power_db'])
    df = df.sort_values(by='frequency').reset_index(drop=True)
    df['power_db'] = pd.to_numeric(df['power_db'], errors='coerce').fillna(-120) # Ganti error dengan nilai rendah
    
    # Perkirakan noise floor sebagai median dari kekuatan sinyal
    noise_floor_estimate = df['power_db'].median()
    
    # Temukan semua puncak (sinyal) yang signifikan di atas noise floor
    peaks, properties = find_peaks(
        df['power_db'], 
        height=noise_floor_estimate + 10, # Sinyal harus 10dB di atas noise
        prominence=prominence, # Seberapa menonjol puncaknya
        width=width # Lebar minimum puncak
    )
    
    # Ekstrak informasi dari setiap sinyal yang terdeteksi
    extracted_signals = []
    for i, peak_idx in enumerate(peaks):
        freq_mhz = df['frequency'][peak_idx] / 1e6
        power_db = df['power_db'][peak_idx]
        # Lebar puncak (dalam index) dikali resolusi frekuensi (5 MHz)
        bandwidth_mhz = properties['widths'][i] * 5 
        
        extracted_signals.append({
            'peak_freq_mhz': round(freq_mhz, 1),
            'peak_power_db': round(power_db, 1),
            'bandwidth_mhz': round(bandwidth_mhz, 1)
        })
        
    return {
        'file': os.path.basename(file_path),
        'noise_floor_db': round(noise_floor_estimate, 1),
        'signals': extracted_signals
    }

### Proses Data Training & Simpan Hasil

Kita sekarang akan menjalankan fungsi ekstraksi fitur pada semua file data training. Hasilnya akan diratakan menjadi sebuah tabel (DataFrame) tunggal dan disimpan sebagai file CSV. File ini akan menjadi input untuk notebook pelatihan model.

In [3]:
# Sama seperti di notebook 01, kita definisikan ulang file training
DATA_PATH = '../data/raw/'
all_files = sorted(glob.glob(os.path.join(DATA_PATH, '*')))
live_files_paths = [
    os.path.join(DATA_PATH, '1700_anomali_1'),
    os.path.join(DATA_PATH, '1000_anomali_4')
]
training_files_paths = [f for f in all_files if f not in live_files_paths]

# Jalankan ekstraksi fitur untuk semua file training
print("Memulai ekstraksi fitur dari data training...")
all_training_features = [extract_physical_features(f) for f in training_files_paths]
print("Ekstraksi fitur selesai.")

Memulai ekstraksi fitur dari data training...
Ekstraksi fitur selesai.


In [4]:
def flatten_features(physical_features_list):
    """Meratakan hasil ekstraksi menjadi format tabel (DataFrame)."""
    flattened_data = []
    for entry in physical_features_list:
        row = {
            'file': entry['file'],
            'noise_floor_db': entry['noise_floor_db'],
            'num_signals': len(entry['signals'])
        }
        # Ambil detail dari sinyal terkuat saja
        if row['num_signals'] > 0:
            strongest_signal = max(entry['signals'], key=lambda x: x['peak_power_db'])
            row['strongest_signal_power_db'] = strongest_signal['peak_power_db']
            row['strongest_signal_freq_mhz'] = strongest_signal['peak_freq_mhz']
            row['strongest_signal_bw_mhz'] = strongest_signal['bandwidth_mhz']
        else: # Handle kasus jika tidak ada sinyal yang terdeteksi
            row['strongest_signal_power_db'] = entry['noise_floor_db']
            row['strongest_signal_freq_mhz'] = 0
            row['strongest_signal_bw_mhz'] = 0
        flattened_data.append(row)
    return pd.DataFrame(flattened_data).set_index('file')

# Buat feature matrix
feature_matrix = flatten_features(all_training_features)

# Simpan ke CSV
PROCESSED_DATA_PATH = '../data/processed/'
if not os.path.exists(PROCESSED_DATA_PATH):
    os.makedirs(PROCESSED_DATA_PATH)
    
output_path = os.path.join(PROCESSED_DATA_PATH, 'training_features.csv')
feature_matrix.to_csv(output_path)

print(f"Feature matrix berhasil disimpan di: {output_path}")
feature_matrix.head()

Feature matrix berhasil disimpan di: ../data/processed/training_features.csv


,noise_floor_db,num_signals,strongest_signal_power_db,strongest_signal_freq_mhz,strongest_signal_bw_mhz
file,,,,,
1000_anomali_1,-47.6,3,-13.9,995.0,13.6
1000_anomali_2,-47.6,3,-13.8,995.0,12.9
1000_anomali_3,-47.6,5,-14.5,1000.0,13.2
1000_anomali_4,-47.6,4,-15.1,995.0,12.4
1500_anomali_1,-47.6,5,-20.4,1495.0,12.3
